# Практическое занятие: N-граммы и принцип скользящего окна

В NLP **N-грамма** — это непрерывная последовательность из $N$ элементов (слов или символов) в данном фрагменте текста.
* $N=1$: **Униграмма** (Unigram) — отдельные слова.
* $N=2$: **Биграмма** (Bigram) — пары соседних слов.
* $N=3$: **Триграмма** (Trigram) — тройки слов.

In [1]:
# Импорт библиотек для анимации и обработки данных
import os
import time
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import collections
import pandas as pd

# Исходная фраза для демонстрации
sentence = "я умер - я проснулся . да , смерть - пробуждение !"
tokens = sentence.split()
num_tokens = len(tokens)

print(f"Длина корпуса: {len(tokens)} экземпляров.")

Длина корпуса: 12 экземпляров.


## Униграммы (N = 1) и частотность слов

Униграммы — это отдельные слова текста. В контексте языковых моделей частота униграммы $Count(w_i)$ служит базой (знаменателем) для расчета условных вероятностей более высокого порядка.


In [2]:
# Извлечение униграмм
unigrams = [tuple([token]) for token in tokens]
unigram_counts = collections.Counter(unigrams)

# Общее количество слов в корпусе (знаменатель для вероятности)
total_words_count = len(tokens)

# Формируем красивую таблицу частот униграмм с расчетом вероятностей
df_uni = pd.DataFrame([
    {
        "Униграмма": f"'{k[0]}'", 
        "Частота (Count)": v,
        "Вероятность P(w_i)": v / total_words_count,
        "Математическая формула": f"Count('{k[0]}') / N = {v} / {total_words_count}"
    } 
    for k, v in unigram_counts.most_common()
])

# Настройка стилей, градиентов и гистограмм внутри таблицы для Jupyter
display(df_uni.style.hide(axis="index")
              .background_gradient(cmap="Blues", subset=["Частота (Count)"])
              .bar(subset=["Вероятность P(w_i)"], color='#accbee')
              .format({"Вероятность P(w_i)": "{:.4f}"})
              .set_caption(f"Частотный словарь униграмм (N=1) | Всего токенов в тексте: {total_words_count}"))



Униграмма,Частота (Count),Вероятность P(w_i),Математическая формула
'я',2,0.1667,Count('я') / N = 2 / 12
'-',2,0.1667,Count('-') / N = 2 / 12
'умер',1,0.0833,Count('умер') / N = 1 / 12
'проснулся',1,0.0833,Count('проснулся') / N = 1 / 12
'.',1,0.0833,Count('.') / N = 1 / 12
'да',1,0.0833,Count('да') / N = 1 / 12
"','",1,0.0833,"Count(',') / N = 1 / 12"
'смерть',1,0.0833,Count('смерть') / N = 1 / 12
'пробуждение',1,0.0833,Count('пробуждение') / N = 1 / 12
'!',1,0.0833,Count('!') / N = 1 / 12


## Биграммы (N = 2) и автоматический расчет условных вероятностей

Биграммная языковая модель оценивает вероятность текущего слова $w_i$, зная только одно предыдущее слово $w_{i-1}$ (Марковское предположение первого порядка):

$$P(w_i \mid w_{i-1}) = \frac{Count(w_{i-1}, w_i)}{Count(w_{i-1})}$$

Ниже представлена функция, которая автоматически выводит математические формулы для всех реально встретившихся в тексте биграмм.


In [3]:
# 1. Извлечение биграмм методом скользящего окна
bigrams = [(tokens[i], tokens[i+1]) for i in range(len(tokens) - 1)]
bigram_counts = collections.Counter(bigrams)

# 2. Автоматическая генерация математических формул-пояснений
print("=== АВТОМАТИЧЕСКИЙ РАСЧЕТ ВЕРОЯТНОСТЕЙ БИГРАММ ===")
for (w_prev, w_curr), b_count in bigram_counts.items():
    u_count = unigram_counts[(w_prev,)]
    prob = b_count / u_count
    
    # Динамическая сборка формулы под конкретные слова
    formula_str = (
        f"P('{w_curr}' | '{w_prev}') = "
        f"Count('{w_prev} {w_curr}') / Count('{w_prev}') = "
        f"{b_count} / {u_count} = {prob:.2f}"
    )
    print(formula_str)

# 3. Построение универсальной матрицы переходов
unique_words = sorted(list(set(tokens)))
prob_matrix_2 = pd.DataFrame(0.0, index=unique_words, columns=unique_words)

for w_prev in unique_words:
    for w_curr in unique_words:
        b_cnt = bigram_counts[(w_prev, w_curr)]
        u_cnt = unigram_counts[(w_prev,)]
        if u_cnt > 0:
            prob_matrix_2.loc[w_prev, w_curr] = b_cnt / u_cnt

# Отображение матрицы в Jupyter
display(prob_matrix_2.style
        .background_gradient(cmap="Blues", axis=None)
        .format("{:.2f}")
        .set_caption("Матрица условных вероятностей Биграмм P(w_i | w_{i-1})"))


=== АВТОМАТИЧЕСКИЙ РАСЧЕТ ВЕРОЯТНОСТЕЙ БИГРАММ ===
P('умер' | 'я') = Count('я умер') / Count('я') = 1 / 2 = 0.50
P('-' | 'умер') = Count('умер -') / Count('умер') = 1 / 1 = 1.00
P('я' | '-') = Count('- я') / Count('-') = 1 / 2 = 0.50
P('проснулся' | 'я') = Count('я проснулся') / Count('я') = 1 / 2 = 0.50
P('.' | 'проснулся') = Count('проснулся .') / Count('проснулся') = 1 / 1 = 1.00
P('да' | '.') = Count('. да') / Count('.') = 1 / 1 = 1.00
P(',' | 'да') = Count('да ,') / Count('да') = 1 / 1 = 1.00
P('смерть' | ',') = Count(', смерть') / Count(',') = 1 / 1 = 1.00
P('-' | 'смерть') = Count('смерть -') / Count('смерть') = 1 / 1 = 1.00
P('пробуждение' | '-') = Count('- пробуждение') / Count('-') = 1 / 2 = 0.50
P('!' | 'пробуждение') = Count('пробуждение !') / Count('пробуждение') = 1 / 1 = 1.00


,!,",",-,.,да,пробуждение,проснулся,смерть,умер,я
!,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
",",0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00
-,0.00,0.00,0.00,0.00,0.00,0.50,0.00,0.00,0.00,0.50
.,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00
да,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
пробуждение,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
проснулся,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00
смерть,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
умер,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
я,0.00,0.00,0.00,0.00,0.00,0.00,0.50,0.00,0.50,0.00


## Триграммы (N = 3) и контекст из двух слов

Триграммная модель учитывает более глубокий контекст. Вероятность слова $w_i$ зависит от двух предшествующих слов $(w_{i-2}, w_{i-1})$:

$$P(w_i \mid w_{i-2}, w_{i-1}) = \frac{Count(w_{i-2}, w_{i-1}, w_i)}{Count(w_{i-2}, w_{i-1})}$$

Контекстом (условием) здесь выступает целая биграмма. Код ниже автоматически находит такие цепочки и строит формулы их предсказания.


In [4]:
# 1. Извлечение триграмм методом скользящего окна
trigrams = [(tokens[i], tokens[i+1], tokens[i+2]) for i in range(len(tokens) - 2)]
trigram_counts = collections.Counter(trigrams)

# 2. Автоматическая генерация математических формул-пояснений для триграмм
print("=== АВТОМАТИЧЕСКИЙ РАСЧЕТ ВЕРОЯТНОСТЕЙ ТРИГРАММ ===")
for (w_prev2, w_prev1, w_curr), t_count in trigram_counts.items():
    # Знаменателем является частота биграммы-предшественника
    b_ctx_count = bigram_counts[(w_prev2, w_prev1)]
    prob_t = t_count / b_ctx_count
    
    # Динамическая сборка формулы
    formula_trigram = (
        f"P('{w_curr}' | '{w_prev2} {w_prev1}') = "
        f"Count('{w_prev2} {w_prev1} {w_curr}') / Count('{w_prev2} {w_prev1}') = "
        f"{t_count} / {b_ctx_count} = {prob_t:.2f}"
    )
    print(formula_trigram)

# 3. Построение матрицы условных вероятностей для триграмм
# Строки матрицы — это пары слов контекста (уникальные биграммы), столбцы — целевые слова (униграммы)
unique_bigrams_ctx = sorted(list(set(bigrams)))
prob_matrix_3 = pd.DataFrame(0.0, index=[f"{b[0]} {b[1]}" for b in unique_bigrams_ctx], columns=unique_words)

for bg in unique_bigrams_ctx:
    for ug in unique_words:
        t_tuple = (bg[0], bg[1], ug)
        t_cnt = trigram_counts[t_tuple]
        b_cnt = bigram_counts[bg]
        
        if b_cnt > 0:
            row_name = f"{bg[0]} bg[1]"
            prob_matrix_3.loc[row_name, ug] = t_cnt / b_cnt

# Отображение триграммной матрицы переходов
display(prob_matrix_3.style
        .background_gradient(cmap="Blues", axis=None)
        .format("{:.2f}")
        .set_caption("Матрица условных вероятностей Триграмм P(w_i | w_{i-2} w_{i-1})"))


=== АВТОМАТИЧЕСКИЙ РАСЧЕТ ВЕРОЯТНОСТЕЙ ТРИГРАММ ===
P('-' | 'я умер') = Count('я умер -') / Count('я умер') = 1 / 1 = 1.00
P('я' | 'умер -') = Count('умер - я') / Count('умер -') = 1 / 1 = 1.00
P('проснулся' | '- я') = Count('- я проснулся') / Count('- я') = 1 / 1 = 1.00
P('.' | 'я проснулся') = Count('я проснулся .') / Count('я проснулся') = 1 / 1 = 1.00
P('да' | 'проснулся .') = Count('проснулся . да') / Count('проснулся .') = 1 / 1 = 1.00
P(',' | '. да') = Count('. да ,') / Count('. да') = 1 / 1 = 1.00
P('смерть' | 'да ,') = Count('да , смерть') / Count('да ,') = 1 / 1 = 1.00
P('-' | ', смерть') = Count(', смерть -') / Count(', смерть') = 1 / 1 = 1.00
P('пробуждение' | 'смерть -') = Count('смерть - пробуждение') / Count('смерть -') = 1 / 1 = 1.00
P('!' | '- пробуждение') = Count('- пробуждение !') / Count('- пробуждение') = 1 / 1 = 1.00


,!,",",-,.,да,пробуждение,проснулся,смерть,умер,я
", смерть",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
- пробуждение,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
- я,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
. да,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
"да ,",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
пробуждение !,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
проснулся .,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
смерть -,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
умер -,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
я проснулся,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


## Расчет результирующей вероятности всей фразы (Chain Rule)

Языковая модель позволяет не только предсказывать следующее слово, но и оценивать общую результирующую вероятность (правдоподобность) целого предложения. Согласно цепному правилу вероятностей (Chain Rule) с Марковским предположением, полная вероятность фразы раскладывается на произведение условных вероятностей:

* **Для биграммной модели:**
$$P(w_1, w_2, \dots, w_n) = P(w_1) \cdot P(w_2 \mid w_1) \cdot P(w_3 \mid w_2) \dots P(w_n \mid w_{n-1})$$

* **Для триграммной модели:**
$$P(w_1, w_2, \dots, w_n) = P(w_1) \cdot P(w_2 \mid w_1) \cdot P(w_3 \mid w_1, w_2) \dots P(w_n \mid w_{n-2}, w_{n-1})$$

In [5]:
# Тестовая фраза для оценки (возьмем кусок из нашего исходного корпуса)
test_phrase = "я проснулся - я умер "
test_tokens = test_phrase.split()

print(f"=== АВТОРАСЧЕТ РЕЗУЛЬТИРУЮЩЕЙ ВЕРОЯТНОСТИ ДЛЯ ФРАЗЫ: '{test_phrase}' ===\n")

# --- 1. Униграммная результирующая вероятность ---
print("1. Униграммная модель (N=1):")
uni_probability = 1.0
steps_uni = []

for w in test_tokens:
    # Вероятность каждого слова берется абсолютно независимо от контекста
    w_count = unigram_counts.get((w,), 0)
    p_word = w_count / total_words_count
    uni_probability *= p_word
    steps_uni.append(f"P('{w}')={p_word:.2f}")

print(" * Сворачивание формулы:", " × ".join(steps_uni))
print(f" * Итоговая вероятность фразы по униграммам: {uni_probability:.7f}\n")


# --- 2. Биграммная результирующая вероятность ---
print("1. Биграммная модель:")
total_words_count = len(tokens)
first_word = test_tokens[0]

# Базовая вероятность первого слова P(w_1)
p_first = unigram_counts.get((first_word,), 0) / total_words_count
bi_probability = p_first
steps_bi = [f"P('{first_word}')={p_first:.2f}"]

for i in range(1, len(test_tokens)):
    w_prev = test_tokens[i-1]
    w_curr = test_tokens[i]
    
    b_cnt = bigram_counts.get((w_prev, w_curr), 0)
    u_cnt = unigram_counts.get((w_prev,), 0)
    
    p_cond = b_cnt / u_cnt if u_cnt > 0 else 0.0
    bi_probability *= p_cond
    steps_bi.append(f"P('{w_curr}'|'{w_prev}')={p_cond:.2f}")

print(" * Сворачивание формулы:", " × ".join(steps_bi))
print(f" * Итоговая вероятность фразы по биграммам: {bi_probability:.5f}\n")


# --- 3. Триграммная результирующая вероятность ---
print("2. Триграммная модель:")
tri_probability = p_first
steps_tri = [f"P('{first_word}')={p_first:.2f}"]

if len(test_tokens) > 1:
    # Для второго слова используем биграммный шаг, так как контекста из двух слов еще нет
    second_word = test_tokens[1]
    b_cnt = bigram_counts.get((first_word, second_word), 0)
    u_cnt = unigram_counts.get((first_word,), 0)
    p_cond_2 = b_cnt / u_cnt if u_cnt > 0 else 0.0
    tri_probability *= p_cond_2
    steps_tri.append(f"P('{second_word}'|'{first_word}')={p_cond_2:.2f}")

for i in range(2, len(test_tokens)):
    w_prev2 = test_tokens[i-2]
    w_prev1 = test_tokens[i-1]
    w_curr = test_tokens[i]
    
    t_cnt = trigram_counts.get((w_prev2, w_prev1, w_curr), 0)
    b_cnt = bigram_counts.get((w_prev2, w_prev1), 0)
    
    p_cond_t = t_cnt / b_cnt if b_cnt > 0 else 0.0
    tri_probability *= p_cond_t
    steps_tri.append(f"P('{w_curr}'|'{w_prev2} {w_prev1}')={p_cond_t:.2f}")

print(" * Сворачивание формулы:", " × ".join(steps_tri))
print(f" * Итоговая вероятность фразы по триграммам: {tri_probability:.5f}")


=== АВТОРАСЧЕТ РЕЗУЛЬТИРУЮЩЕЙ ВЕРОЯТНОСТИ ДЛЯ ФРАЗЫ: 'я проснулся - я умер ' ===

1. Униграммная модель (N=1):
 * Сворачивание формулы: P('я')=0.17 × P('проснулся')=0.08 × P('-')=0.17 × P('я')=0.17 × P('умер')=0.08
 * Итоговая вероятность фразы по униграммам: 0.0000322

1. Биграммная модель:
 * Сворачивание формулы: P('я')=0.17 × P('проснулся'|'я')=0.50 × P('-'|'проснулся')=0.00 × P('я'|'-')=0.50 × P('умер'|'я')=0.50
 * Итоговая вероятность фразы по биграммам: 0.00000

2. Триграммная модель:
 * Сворачивание формулы: P('я')=0.17 × P('проснулся'|'я')=0.50 × P('-'|'я проснулся')=0.00 × P('я'|'проснулся -')=0.00 × P('умер'|'- я')=0.00
 * Итоговая вероятность фразы по триграммам: 0.00000


## Генерация текста на основе Марковских цепей

Зная условные вероятности, мы можем запустить процесс автодополнения (генерации) текста. Начиная со слова-затравки (seed), модель смотрит на текущий контекст, выбирает распределение вероятностей для следующего слова и делает случайный выбор (сэмплирование) с учетом этих весов. 

In [6]:
import random

def generate_text_bigram(start_word, max_length=10):
    """Генерация текста по Марковской цепи 1-го порядка (Биграммы)"""
    current_word = start_word
    generated_words = [current_word]
    
    for _ in range(max_length - 1):
        # Ищем все возможные продолжения для текущего слова
        candidates = [bg[1] for bg in bigram_counts.keys() if bg[0] == current_word]
        
        if not candidates:
            break # Если слово тупиковое и после него в корпусе ничего не шло
            
        # Считаем веса (частоты) для каждого кандидата
        weights = [bigram_counts[(current_word, cand)] for cand in candidates]
        
        # Случайный выбор следующего слова с учетом весов
        next_word = random.choices(candidates, weights=weights)[0]
        generated_words.append(next_word)
        current_word = next_word
        
    return " ".join(generated_words)


def generate_text_trigram(start_context, max_length=10):
    """Генерация текста по Марковской цепи 2-го порядка (Триграммы)"""
    # На вход ожидается строка из двух слов, например: "мама мыла"
    context_tokens = start_context.split()
    if len(context_tokens) < 2:
        return "Ошибка: Для триграммной генерации нужен контекст минимум из 2 слов!"
        
    w_prev2, w_prev1 = context_tokens[0], context_tokens[1]
    generated_words = [w_prev2, w_prev1]
    
    for _ in range(max_length - 2):
        # Ищем продолжения, где первые два слова совпадают с текущим контекстом
        candidates = [tg[2] for tg in trigram_counts.keys() if tg[0] == w_prev2 and tg[1] == w_prev1]
        
        if not candidates:
            # Если точной триграммы нет, модель "откатывается" (backoff) на биграммный поиск по последнему слову
            candidates = [bg[1] for bg in bigram_counts.keys() if bg[0] == w_prev1]
            if not candidates:
                break
            weights = [bigram_counts[(w_prev1, cand)] for cand in candidates]
        else:
            weights = [trigram_counts[(w_prev2, w_prev1, cand)] for cand in candidates]
            
        next_word = random.choices(candidates, weights=weights)[0]
        generated_words.append(next_word)
        
        # Сдвигаем контекстное окно вперед
        w_prev2, w_prev1 = w_prev1, next_word
        
    return " ".join(generated_words)


In [7]:

# --- Демонстрация работы генераторов ---
print("=== ТЕСТИРОВАНИЕ АВТОМАТИЧЕСКОГО ПРОДОЛЖЕНИЯ ФРАЗ ===\n")

seed_word = "я проснулся - я умер"
print(f"Биграммное продолжение для слова '{seed_word}':")
print(f" -> '{generate_text_bigram(start_word=seed_word, max_length=7)}'\n")

seed_context = "я проснулся"
print(f"Триграммное продолжение для контекста '{seed_context}':")
print(f" -> '{generate_text_trigram(start_context=seed_context, max_length=7)}'")

=== ТЕСТИРОВАНИЕ АВТОМАТИЧЕСКОГО ПРОДОЛЖЕНИЯ ФРАЗ ===

Биграммное продолжение для слова 'я проснулся - я умер':
 -> 'я проснулся - я умер'

Триграммное продолжение для контекста 'я проснулся':
 -> 'я проснулся . да , смерть -'
